In [1]:
from functools import partial, reduce
from pathlib import Path
import pandas as pd
import numpy as np
import time

route = Path.home() / "OneDrive" / "Rawdata"

if not hasattr(pd, "_codex_original_read_csv"):
    pd._codex_original_read_csv = pd.read_csv
if not hasattr(pd, "_codex_original_read_excel"):
    pd._codex_original_read_excel = pd.read_excel

_pd_read_csv = pd._codex_original_read_csv
_pd_read_excel = pd._codex_original_read_excel

def _retry_io(func, *args, max_retries=3, retry_delay=1.0, **kwargs):
    last_error = None
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except OSError as exc:
            last_error = exc
            if attempt == max_retries - 1:
                raise
            time.sleep(retry_delay)
    raise last_error

def read_csv_retry(*args, **kwargs):
    return _retry_io(_pd_read_csv, *args, **kwargs)

def read_excel_retry(*args, **kwargs):
    return _retry_io(_pd_read_excel, *args, **kwargs)

pd.read_csv = read_csv_retry
pd.read_excel = read_excel_retry



In [2]:
countrycode = pd.read_excel(
    route / "Country Code" / "Countrycode.xlsx",
    sheet_name="Sheet1",
    na_values="..",
)
code_alpha3 = countrycode[["Numeric", "Alpha-3 code"]]
code_wb = countrycode[["Numeric", "WBCode"]]
code_CN = countrycode[["Numeric", "CountryName_CN"]]

In [3]:
# WDI_World Development Index
wb = pd.read_csv(
    route / "World Bank World Development Index" / "WDI_csv" / "WDIData.csv",
    na_values="..",
)
wb = (
    wb.drop(columns=["Country Name", "Indicator Code"])
    .rename(columns={"Country Code": "Alpha-3 code"})
    .merge(code_alpha3, how="right", on=["Alpha-3 code"])
    .drop(columns=["Alpha-3 code"])
    .melt(id_vars=("Numeric", "Indicator Name"))
    .astype({"Numeric": "float64", "variable": "int64", "value": "float64"})
    .pivot_table(
        columns="Indicator Name",
        values="value",
        index=["Numeric", "variable"],
        dropna="..",
    )
    .reset_index()
    .rename(columns={"variable": "Year"})
)

In [4]:
# FAO SDG data
fao_path = max(
    (route / "FAO Data").glob("fao-sdg-sdmx-data*.csv"),
    key=lambda p: p.stat().st_mtime,
)
fao_raw_file = fao_path.relative_to(route).as_posix()

fao = pd.read_csv(
    fao_path,
    na_values="..",
    low_memory=False,
)[["AREA", "REF_AREA", "TIME_PERIOD", "SERIES_DESC", "OBS_VALUE"]]

fao = fao[fao["REF_AREA"].isin(countrycode["Numeric"])]

fao["OBS_VALUE"] = fao["OBS_VALUE"].apply(pd.to_numeric, errors="coerce")
FAO = (
    fao.pivot_table(
        columns="SERIES_DESC", values="OBS_VALUE", index=["REF_AREA", "TIME_PERIOD"]
    )
    .reset_index()
    .rename(columns={"TIME_PERIOD": "Year", "REF_AREA": "Numeric"})
)


In [5]:
# EPI_Environmental Performance Index
def epi(path, max_retries=3, retry_delay=1.0):
    import time

    last_error = None
    for attempt in range(max_retries):
        try:
            data = pd.read_csv(path)
            break
        except OSError as exc:
            last_error = exc
            if attempt == max_retries - 1:
                raise
            time.sleep(retry_delay)

    epi_var = path.stem.split("_")[0]
    if "CXN.mry" in data.columns:
        data = data.drop("CXN.mry", axis=1)
    if "WST.mry" in data.columns:
        data = data.drop("WST.mry", axis=1)
    data = data[data["code"].isin(countrycode["Numeric"])].rename(
        columns={"{}.raw.{}".format(epi_var, i): str(i) for i in range(1700, 2046)}
    )
    data.index = data.code
    data = data.drop(["iso", "country", "code"], axis=1).stack()
    data = data.rename(epi_var)
    return data


df = pd.DataFrame({"Alpha-3 code": [], "Year": []})
for p in sorted((
    route
    / "EPI Environmental Performance Index"
    / "Raw"
).glob("*_raw_na.csv")):
    data = epi(p)
    data = data.reset_index().rename(columns={"level_1": "Year", "code": "Numeric"})
    df = df.merge(data, how="outer")
epivariables = pd.read_csv(
    route / "EPI Environmental Performance Index" / "epi2024variables2024-12-11.csv"
)
official_epi_map = dict(
    (k["Abbreviation"], k["Variable"]) for _, k in epivariables.iterrows()
)
legacy_epi_map = {
    "BLC": "Black carbon emissions (BLC)",
    "CDO": "Carbon dioxide emissions (CDO)",
    "CH4": "Methane emissions (CH4)",
    "FOG": "Fluorinated gas emissions (FOG)",
    "GDP": "Gross domestic product (GDP)",
    "GHA": "GHG growth rate adjusted by economic output (GHA)",
    "GHG": "Greenhouse gas emissions (GHG)",
    "GHI": "Greenhouse gas emissions intensity (GHI)",
    "GHP": "Greenhouse gas emissions per capita (GHP)",
    "GOE": "GHG growth rate adjusted by per-capita emissions (GOE)",
    "GPC": "Gross domestic product per capita (GPC)",
    "HDI": "Human Development Index (HDI)",
    "HFX": "Household solid fuels share (HFX)",
    "IF5": "Industrial fishing pressure (IF5)",
    "IFA": "Industrial fishing activity (IFA)",
    "IFC": "Industrial fishing catch (IFC)",
    "LUE": "Land-use emissions (LUE)",
    "NCR": "Nutrient circularity ratio (NCR)",
    "NOE": "Nitrous oxide emissions intensity (NOE)",
    "NOT": "Nitrous oxide emissions (NOT)",
    "NOX": "Nitrogen oxides emissions (NOX)",
    "NRY": "Relative crop yield ratio (NRY)",
    "NTI": "Nitrogen balance indicator (NTI)",
    "NTL": "Nitrogen total load (NTL)",
    "NUE": "Nitrogen use efficiency (NUE)",
    "OZE": "Ozone exposure index (OZE)",
    "PCR": "Pesticide contamination risk (PCR)",
    "PDN": "Population density (PDN)",
    "PF5": "PM2.5 fossil-fuel exposure (PF5)",
    "PFA": "Fishing activity pressure (PFA)",
    "PFC": "Fishing catch pressure (PFC)",
    "PMD": "PM2.5 disease burden (PMD)",
    "PME": "PM2.5 exposure (PME)",
    "POP": "Population (POP)",
    "PTI": "Phosphorus balance indicator (PTI)",
    "PUE": "Phosphorus use efficiency (PUE)",
    "ROL": "Rule of law (ROL)",
    "RQU": "Regulatory quality (RQU)",
    "SO2": "Sulfur dioxide emissions (SO2)",
    "TC5": "Climate target pathway indicator (TC5)",
    "TCA": "Projected cumulative emissions to 2050 (TCA)",
    "TCC": "Country-target carbon trajectory (TCC)",
    "TCL": "Land-cover carbon flux (TCL)",
    "URB": "Urban population share (URB)",
    "VOA": "Voice and accountability (VOA)",
    "WHR": "Waste recovery rate (WHR)",
    "WRE": "Wastewater reused (WRE)",
}
epi_cleanup_map = {
    "Forest Lanscape Integrity": "Forest Landscape Integrity",
    "Anthropogenic PM2.5 exposure ": "Anthropogenic PM2.5 exposure",
    "Waste Generated Per Capita": "Waste generated per capita",
}
epi_rename_map = {**official_epi_map, **legacy_epi_map}
epi_raw_indicator_map = {
    new_name: old_name for old_name, new_name in epi_rename_map.items()
}
for old_name, new_name in epi_cleanup_map.items():
    if old_name in epi_raw_indicator_map:
        epi_raw_indicator_map[new_name] = epi_raw_indicator_map.pop(old_name)

EPI = df
EPI["Year"] = EPI["Year"].apply(pd.to_numeric)
EPI = EPI.rename(columns=epi_rename_map).rename(columns=epi_cleanup_map).drop(columns=["Alpha-3 code"], errors="ignore")



In [6]:
#EIA_Energy Information Administration_US
duplicated_series = ["INTL.4006-8-MMTCD.A", "INTL.4002-8-MMTCD.A"]
former_countries_or_EIA_specific_regions = [
    "WLD",
    "XKS",
    "HITZ",
    "USIQ",
    "DEUW",
    "USOH",
    "NLDA",
    "CSK",
    "DDR",
    "WAK",
    "YUG",
    "SCG",
    "SUN",
]
eia_series_name_cleaner = (
    r"(?P<Variables>Crude oil, NGPL, and other liquids production|"
    r"Total energy consumption from nuclear, renewables, and other|"
    r"Total energy production from nuclear, renewables, and other|"
    r"Solar, tide, wave, fuel cell electricity .+?|"
    r".+?), (?P<region>Gambia, The|.+), Annual"
)
from zipfile import ZipFile

with ZipFile(route / "EIA Energy Information Administration" / "INTL.zip") as zf:
    with zf.open("INTL.txt") as fh:
        EIA_INTL = pd.read_json(fh, lines=True)
EIA_INTL.dropna(subset=["series_id", "geoset_id", "data"], inplace=True)
for i in range(11, 28):
    EIA_INTL.query(f"not series_id.str.contains('WP{i}')", inplace=True)
EIA_INTL.query("not series_id.str.contains('-OPSA-')", inplace=True)
EIA_INTL.query("geoset_id not in @duplicated_series", inplace=True)
EIA_INTL.query(
    "geography not in @former_countries_or_EIA_specific_regions", inplace=True
)
EIA_INTL.query("f == 'A'", inplace=True)
EIA_INTL = EIA_INTL.explode("data").reset_index(drop=True)
EIA_INTL[["Year", "value"]] = (
    pd.DataFrame(
        EIA_INTL["data"].tolist(), index=EIA_INTL.index, columns=["Year", "value"]
    )
    .replace({"value": {"--": None, "w": np.nan, "NA": np.nan, "ie": np.nan}})
    .astype({"Year": "int64", "value": "float64"})
)
EIA_INTL = (
    EIA_INTL.drop(columns=["data"])
    .query(r"not geography.str.contains('\+')")[["name", "geography", "Year", "value", "units"]]
    .replace(eia_series_name_cleaner, r"\g<Variables>", regex=True)
)
EIA_INTL["Variables"] = EIA_INTL["name"] + " (" + EIA_INTL["units"] + ")"
EIA_INTL = (
    EIA_INTL.drop(["name", "units"], axis=1)
    .pivot(index=["geography", "Year"], columns="Variables", values="value")
    .reset_index()
    .merge(code_alpha3, left_on="geography", right_on="Alpha-3 code")
)
EIA_INTL




,geography,Year,Anthracite consumption (1000 metric tons),Anthracite consumption (million metric tons of oil equivalent),Anthracite consumption (quadrillion Btu),Anthracite consumption (terajoules),Anthracite consumption (thousand short tons),Anthracite exports (1000 metric tons),Anthracite exports (million metric tons of oil equivalent),Anthracite exports (quadrillion Btu),...,Total energy production from renewables and other (quadrillion Btu),Vented and flared natural gas production (billion cubic feet),Vented and flared natural gas production (billion cubic meters),Wind electricity installed capacity (million kilowatts),Wind electricity net generation (billion kilowatthours),Wind electricity net generation (million metric tons of oil equivalent),Wind electricity net generation (quadrillion Btu),Wind electricity net generation (terajoules),Numeric,Alpha-3 code
0,AGO,1973,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,AGO
1,AGO,1974,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,AGO
2,AGO,1975,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,AGO
3,AGO,1976,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,AGO
4,AGO,1977,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,AGO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627,ZWE,2021,1.521545,0.001033,0.000041,43.253484,1.677217,0.00000,0.000000e+00,0.000000e+00,...,0.023090,NaN,NaN,0.0,0.0,0.0,0.0,0.0,716,ZWE
2628,ZWE,2022,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000e+00,0.000000e+00,...,0.023100,NaN,NaN,0.0,0.0,0.0,0.0,0.0,716,ZWE
2629,ZWE,2023,0.000000,0.000000,0.000000,0.000000,0.000000,0.00009,5.670673e-08,2.250305e-09,...,0.021954,NaN,NaN,0.0,0.0,0.0,0.0,0.0,716,ZWE
2630,ZWE,2024,0.000000,0.000000,0.000000,0.000000,0.000000,0.00020,1.260151e-07,5.000683e-09,...,0.022804,NaN,NaN,0.0,0.0,0.0,0.0,0.0,716,ZWE


In [7]:
### Ourworldindata education
data_path = route / "Ourworldindata" / "Education and Work Data v2.1.csv"
name_path = route / "Ourworldindata" / "Education and Work Data v2.1.description_short.csv"

# 真正的数据
educationyears_raw = pd.read_csv(data_path)
educationyears_raw = educationyears_raw[
    educationyears_raw["isocode3"].isin(code_alpha3["Alpha-3 code"])
].copy()

# 只保留国家层级
educationyears_raw = educationyears_raw.query('`level` == "National"').copy()

# 只读取第二个文件的表头，用来改变量名
name_header = pd.read_csv(name_path, nrows=0)
rename_map = dict(zip(educationyears_raw.columns, name_header.columns))

# 把原始变量代码改成 description_short
educationyears_raw = educationyears_raw.rename(columns=rename_map)

# 年份列改成统一名字
if "Year of data collection" in educationyears_raw.columns:
    educationyears_raw = educationyears_raw.rename(columns={"Year of data collection": "Year"})

# 国家代码列改成统一名字
if "ISO country code 3 digit" in educationyears_raw.columns:
    educationyears_raw = educationyears_raw.rename(columns={"ISO country code 3 digit": "Alpha-3 code"})

educationyears_raw["Year"] = pd.to_numeric(educationyears_raw["Year"], errors="coerce")
educationyears_raw = educationyears_raw.merge(code_alpha3, how="left", on=["Alpha-3 code"])

education_meta_cols = {
    "Numeric",
    "Year",
    "Alpha-3 code",
    "Country name",
    "Continent",
    "Source of data",
    "Global Data Lab region code",
    "Aggregation level",
    "Sub-national region name",
}
education_value_cols = [c for c in educationyears_raw.columns if c not in education_meta_cols]

educationyears_long = (
    educationyears_raw
    .melt(
        id_vars=["Numeric", "Year", "Source of data"],
        value_vars=education_value_cols,
        var_name="variable",
        value_name="value",
    )
    .rename(columns={"Source of data": "sub_source"})
)
educationyears_long["sub_source"] = educationyears_long["sub_source"].fillna("Unknown")
educationyears_long["value"] = pd.to_numeric(educationyears_long["value"], errors="coerce")
educationyears_long = educationyears_long.dropna(subset=["value"])

educationyears = (
    educationyears_long
    .pivot_table(
        index=["Numeric", "Year"],
        columns="variable",
        values="value",
        aggfunc="mean",
    )
    .reset_index()
)
educationyears

variable,Numeric,Year,Educational attendance boys 12-14,Educational attendance boys 15-17,Educational attendance boys 18-20,Educational attendance boys 21-23,Educational attendance boys 6-8,Educational attendance boys 9-11,Educational attendance children 12-14,Educational attendance children 15-17,...,Number of Mean years education of women aged 25+,Number of Mean years education of women aged 40-59,Number of Mean years education of women aged 60+,Percentage of employed men in agriculture,Percentage of employed men in lower nonfarm jobs,Percentage of employed men in upper nonfarm jobs,Percentage of employed women in agriculture,Percentage of employed women in lower nonfarm jobs,Percentage of employed women in upper nonfarm jobs,Percentage of women in paid employment
0,12,2002,91.72,57.92,23.31,11.96,97.67,98.27,88.76,58.33,...,13666.0,4676.0,2216.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12,2013,95.01,74.08,41.97,20.98,97.08,98.52,94.25,76.51,...,38739.0,13845.0,5935.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12,2019,95.31,73.55,40.50,24.07,96.24,98.02,95.05,78.05,...,41292.0,16597.0,8204.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,24,2000,81.13,69.15,42.41,0.00,59.69,79.52,76.94,60.91,...,5266.0,1667.0,516.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,24,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5599.0,1750.0,353.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,894,1996,75.90,62.04,34.25,12.44,34.46,73.34,74.74,53.67,...,6538.0,2239.0,779.0,48.849,43.556,7.595,41.209,54.232,4.559,47.424
211,894,2002,78.92,68.28,40.24,14.14,37.83,76.52,77.58,59.97,...,6274.0,2003.0,853.0,54.583,38.285,7.133,63.934,32.508,3.558,56.025
212,894,2007,91.08,87.99,54.34,21.33,51.55,87.71,90.26,78.51,...,6216.0,1966.0,853.0,44.927,45.924,9.149,47.897,46.323,5.781,48.464
213,894,2014,88.14,79.77,48.50,20.93,52.75,88.34,88.42,75.43,...,14655.0,4725.0,2086.0,51.046,40.778,8.176,49.555,44.513,5.933,51.077


In [8]:
# Global Data Lab
gdl_raw = pd.read_excel(route / "GDL Global Data Lab" / "GDL-Mean-years-schooling-data.xlsx")
gdl_years = sorted(col for col in gdl_raw.columns if isinstance(col, (int, np.integer)))

GDL_source = (
    gdl_raw[["ISO_Code", "Level", *gdl_years]]
    .query('Level == "National"')
    .rename(columns={"ISO_Code": "Alpha-3 code"})
    .merge(code_alpha3, how="inner", on=["Alpha-3 code"])
)

GDL_long = (
    GDL_source
    .drop(columns=["Alpha-3 code", "Level"])
    .melt(id_vars=["Numeric"], var_name="Year", value_name="value")
    .assign(variable="Mean years schooling")
)
GDL_long["Year"] = pd.to_numeric(GDL_long["Year"], errors="coerce")
GDL_long["value"] = pd.to_numeric(GDL_long["value"], errors="coerce")
GDL_long = GDL_long.dropna(subset=["value"])

GDL = (
    GDL_long
    .pivot_table(
        index=["Numeric", "Year"],
        columns="variable",
        values="value",
        aggfunc="first",
    )
    .reset_index()
)
GDL


variable,Numeric,Year,Mean years schooling
0,12,1990,8.165
1,12,1991,8.187
2,12,1992,8.258
3,12,1993,8.263
4,12,1994,8.322
...,...,...,...
1671,894,2019,9.180
1672,894,2020,9.182
1673,894,2021,9.184
1674,894,2022,9.184


In [9]:
# UNU-WIDER GINI
gini = (
    pd.read_excel(
        route / "UNU-WIDER World Income Inequality Database" / "WIID-29APR2025.xlsx"
    )[["c3", "year", "gini"]]
    .rename(columns={"gini": "gini", "c3": "Alpha-3 code", "year": "Year"})
    .merge(code_alpha3, how="right", on=["Alpha-3 code"])[
        ["Year", "Numeric", "gini", "Alpha-3 code"]
    ]
)

In [10]:
import re

imf_skip_files = (
    0,  # summary
    1,  # not annual data
    8,  # bilateral trade
    10,  # bilateral trade
    13,  # handled separately in old workflow
    16,  # future projections
    17,  # future projections
    26,  # only world data in old workflow
    27,  # only world data in old workflow
)
imf_transform_note = (
    "Constructed variable labels by joining the non-constant descriptor columns "
    "within each IMF-CID source file after removing columns that are constant "
    "inside the file; year columns are standardized to four-digit labels."
)

imf_long_parts = []
for csv_path in sorted((route / "IMF" / "IMF-CID").glob("*.csv")):
    i = int(csv_path.stem.split("_", 1)[0])
    if i in imf_skip_files:
        continue

    df = pd.read_csv(csv_path)
    years = [int(col) for col in df.columns if re.fullmatch(r"\d{4}", str(col))]
    if not years:
        continue

    nunique = df.nunique()
    cols_to_drop = nunique[nunique <= 1].index
    df = (
        df.drop(columns=cols_to_drop)
        .rename(columns={str(year): year for year in years})
        .rename(columns={"ISO3": "Alpha-3 code"})
        .dropna(subset=["Alpha-3 code"])
        .merge(code_alpha3, how="inner", on=["Alpha-3 code"])
    )

    descriptor_cols = [
        col
        for col in df.columns
        if col not in ("Country", "ISO2", "Alpha-3 code", "Numeric", *years)
    ]
    if descriptor_cols:
        df["variable"] = pd.Series(
            (
                "|".join(row)
                for row in df[descriptor_cols].fillna("").astype(str).itertuples(index=False, name=None)
            ),
            index=df.index,
        ).str.strip("|").str.replace(r"\|+", "|", regex=True)
    else:
        df["variable"] = csv_path.stem

    if "Indicator" in df.columns:
        raw_indicator = df["Indicator"].fillna("")
    else:
        raw_indicator = pd.Series("", index=df.index)
    df["raw_indicator_name"] = raw_indicator.replace("", pd.NA).fillna(df["variable"])

    imf_long = df.melt(
        id_vars=["Numeric", "Alpha-3 code", "variable", "raw_indicator_name"],
        value_vars=years,
        var_name="Year",
        value_name="value",
    )
    imf_long["Year"] = pd.to_numeric(imf_long["Year"], errors="coerce")
    imf_long["value"] = pd.to_numeric(imf_long["value"], errors="coerce")
    imf_long["raw_file"] = csv_path.relative_to(route).as_posix()
    imf_long_parts.append(imf_long)

IMF_CID_long = pd.concat(imf_long_parts, ignore_index=True).dropna(subset=["value"])
IMF_CID = (
    IMF_CID_long.pivot_table(
        index=["Numeric", "Year"],
        columns="variable",
        values="value",
        aggfunc="first",
    )
    .reset_index()
)

# Merge all the data
EIA_INTL_panel = EIA_INTL.drop(columns=["geography", "Alpha-3 code"])

dfs = [
    wb,
    EPI,
    FAO,
    EIA_INTL_panel,
    IMF_CID,
    GDL,
    educationyears,
    gini
]
df_final = reduce(
    lambda left, right: pd.merge(left, right, how="outer", on=("Year", "Numeric")), dfs
)
df_final = (
    df_final.merge(countrycode, how="inner", on=["Numeric"], suffixes=("", "_DROP"))
    .filter(regex="^(?!.*_DROP)")
    .fillna({"wardummy": 0})
)

current_account_alias = 'Current account balance, percent of GDP (Percent of GDP)(IMF)'
current_account_note = (
    'Alias of the WDI current-account-balance-as-percent-of-GDP series kept with the thesis-era IMF wording '
    'for backward compatibility; WDI metadata attributes the source to IMF Balance of Payments Statistics.'
)
if current_account_alias not in df_final.columns and 'Current account balance (% of GDP)' in df_final.columns:
    df_final[current_account_alias] = df_final['Current account balance (% of GDP)']
current_account_panel = (
    df_final[['Numeric', 'Year', current_account_alias]]
    .dropna(subset=[current_account_alias])
    .copy()
)


wetland_proxy_components = [
    'Shrubs and/or herbaceous vegetation, aquatic or regularly flooded|1000 HA|ECCCV|Shrubs and/or Herbaceous Vegetation and Aquatic Or Regularly Flooded|Environment, Climate Change, Climate and Weather, Land Cover Accounts, Shrubs and/or Herbaceous Vegetation and Aquatic Or Regularly Flooded|Climate regulating',
    'Mangroves|1000 HA|ECCCM|Mangroves|Environment, Climate Change, Climate and Weather, Land Cover Accounts, Mangroves|Climate regulating',
]
wetland_proxy_available = [c for c in wetland_proxy_components if c in df_final.columns]
wetland_proxy_note = (
    'Proxy for wetland area share constructed from FAOSTAT Land Cover classes '
    'available via IMF-CID: shrubs and/or herbaceous vegetation, aquatic or regularly flooded '
    'plus mangroves when available; converted from 1000 HA to square kilometers and divided by land area.'
)
if wetland_proxy_available and 'Land area (sq. km)' in df_final.columns:
    wetland_area_sqkm = df_final[wetland_proxy_available].fillna(0).sum(axis=1) * 10
    df_final['Wetland area（% of land area)'] = (wetland_area_sqkm / df_final['Land area (sq. km)']) * 100
else:
    df_final['Wetland area（% of land area)'] = pd.NA
wetland_proxy = (
    df_final[['Numeric', 'Year', 'Wetland area（% of land area)']]
    .dropna(subset=['Wetland area（% of land area)'])
    .copy()
)

def get_value_columns(df, extra_exclude=None):
    excluded = {"Numeric", "Year", "Alpha-3 code"}
    if extra_exclude:
        excluded |= set(extra_exclude)
    return [c for c in df.columns if c not in excluded]

dataset_specs = [
    (wb, "WDI", "World Bank", "World Bank World Development Index/WDI_csv/WDIData.csv", ""),
    (EPI, "EPI", "Yale Environmental Performance Index", "EPI Environmental Performance Index/Raw + epi2024variables2024-12-11.csv", "Readable labels use the official 2024 variable table where available; legacy raw-code variables are renamed to readable labels and keep the original code in raw_indicator_name."),
    (FAO, "FAO SDG", "FAO", fao_raw_file, "Latest FAO SDG extract selected automatically from the FAO Data directory and reshaped from SDMX long format to a country-year wide panel using SERIES_DESC labels."),    
    (EIA_INTL_panel, "EIA International Energy", "U.S. Energy Information Administration", "EIA Energy Information Administration/INTL.zip", "Parsed annual country series from EIA INTL; variable labels are constructed from the cleaned series name plus units after removing trailing geography from the source title."),
    (GDL, "Global Data Lab", "Global Data Lab", "GDL Global Data Lab/GDL-Mean-years-schooling-data.xlsx", "Filtered to national-level observations and reshaped to a country-year panel for mean years of schooling."),
    (educationyears, "OWID Education and Work", "Our World in Data", "Ourworldindata/Education and Work Data v2.1.csv", "Filtered to SSA and National; columns renamed using description_short; wide table aggregates multiple Source of data rows to country-year means."),
    (gini, "WIID Gini", "UNU-WIDER", "UNU-WIDER World Income Inequality Database/WIID-29APR2025.xlsx", ""),
    (wetland_proxy, "FAOSTAT Land Cover proxy", "Food and Agriculture Organization (via IMF-CID)", "IMF/IMF-CID/28_Land_Cover_Accounts.csv", wetland_proxy_note),
    (current_account_panel, "WDI", "World Bank", "World Bank World Development Index/WDI_csv/WDIData.csv", current_account_note),
]

dataset_raw_name_maps = {
    "EPI": epi_raw_indicator_map,
}

variable_source_rows = []
for df_obj, dataset_name, source_org, raw_file, transform_note in dataset_specs:
    raw_name_map = dataset_raw_name_maps.get(dataset_name, {})
    sub_source_note = ""
    if dataset_name == "OWID Education and Work":
        sub_source_note = "See raw_long.sub_source for DHS, MICS, and other source labels."

    for variable in get_value_columns(df_obj):
        variable_source_rows.append(
            {
                "variable": variable,
                "dataset_name": dataset_name,
                "source_org": source_org,
                "raw_file": raw_file,
                "raw_indicator_name": raw_name_map.get(variable, variable),
                "transform_note": transform_note,
                "key": "Numeric-Year",
                "sub_source_note": sub_source_note,
            }
        )

imf_variable_source_rows = (
    IMF_CID_long[["variable", "raw_file", "raw_indicator_name"]]
    .drop_duplicates()
    .assign(
        dataset_name="IMF Climate Change Indicators",
        source_org="International Monetary Fund",
        transform_note=imf_transform_note,
        key="Numeric-Year",
        sub_source_note="",
    )
)
variable_source_rows.extend(imf_variable_source_rows.to_dict("records"))

variable_source_dict = (
    pd.DataFrame(variable_source_rows)
    .drop_duplicates()
    .sort_values(["dataset_name", "variable"])
    .reset_index(drop=True)
)

raw_long_parts = []
for df_obj, dataset_name, source_org, raw_file, transform_note in dataset_specs:
    raw_name_map = dataset_raw_name_maps.get(dataset_name, {})
    if dataset_name == "OWID Education and Work":
        long_df = educationyears_long.copy()
    elif dataset_name == "Global Data Lab":
        long_df = GDL_long.copy()
        long_df["sub_source"] = pd.NA
    else:
        value_cols = get_value_columns(df_obj)
        long_df = df_obj.melt(
            id_vars=["Numeric", "Year"],
            value_vars=value_cols,
            var_name="variable",
            value_name="value",
        )
        long_df["sub_source"] = pd.NA

    long_df["dataset_name"] = dataset_name
    long_df["source_org"] = source_org
    long_df["raw_file"] = raw_file
    long_df["raw_indicator_name"] = long_df["variable"].map(lambda x: raw_name_map.get(x, x))
    long_df["transform_note"] = transform_note
    raw_long_parts.append(long_df)

imf_raw_long = IMF_CID_long[["Numeric", "Year", "variable", "value", "raw_file", "raw_indicator_name"]].copy()
imf_raw_long["sub_source"] = pd.NA
imf_raw_long["dataset_name"] = "IMF Climate Change Indicators"
imf_raw_long["source_org"] = "International Monetary Fund"
imf_raw_long["transform_note"] = imf_transform_note
raw_long_parts.append(imf_raw_long)

raw_long = pd.concat(raw_long_parts, ignore_index=True)
raw_long = raw_long.dropna(subset=["value"])
raw_long["Year"] = pd.to_numeric(raw_long["Year"], errors="coerce")
raw_long = (
    raw_long
    .merge(countrycode, how="left", on=["Numeric"], suffixes=("", "_DROP"))
    .filter(regex="^(?!.*_DROP)")
    .sort_values(["Numeric", "Year", "dataset_name", "variable"])
    .reset_index(drop=True)
)

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

df_final.to_csv(project_dir / "df_final.csv", encoding='utf-8-sig')

variable_source_dict.to_csv(project_dir / "variable_source_dict.csv", index=False, encoding='utf-8-sig')

variable_source_dict.head()


/var/folders/6v/nc250lnx4qs4c1hjd2jt44_00000gn/T/ipykernel_26934/3927195364.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final[current_account_alias] = df_final['Current account balance (% of GDP)']
/var/folders/6v/nc250lnx4qs4c1hjd2jt44_00000gn/T/ipykernel_26934/3927195364.py:133: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['Wetland area（% of land area)'] = (wetland_area_sqkm / df_final['Land area (sq. km)']) * 100


,variable,dataset_name,source_org,raw_file,raw_indicator_name,transform_note,key,sub_source_note
0,Anthracite consumption (1000 metric tons),EIA International Energy,U.S. Energy Information Administration,EIA Energy Information Administration/INTL.zip,Anthracite consumption (1000 metric tons),Parsed annual country series from EIA INTL; va...,Numeric-Year,
1,Anthracite consumption (million metric tons of...,EIA International Energy,U.S. Energy Information Administration,EIA Energy Information Administration/INTL.zip,Anthracite consumption (million metric tons of...,Parsed annual country series from EIA INTL; va...,Numeric-Year,
2,Anthracite consumption (quadrillion Btu),EIA International Energy,U.S. Energy Information Administration,EIA Energy Information Administration/INTL.zip,Anthracite consumption (quadrillion Btu),Parsed annual country series from EIA INTL; va...,Numeric-Year,
3,Anthracite consumption (terajoules),EIA International Energy,U.S. Energy Information Administration,EIA Energy Information Administration/INTL.zip,Anthracite consumption (terajoules),Parsed annual country series from EIA INTL; va...,Numeric-Year,
4,Anthracite consumption (thousand short tons),EIA International Energy,U.S. Energy Information Administration,EIA Energy Information Administration/INTL.zip,Anthracite consumption (thousand short tons),Parsed annual country series from EIA INTL; va...,Numeric-Year,


In [11]:
{
    "df_final_shape": df_final.shape,
    "variable_source_dict_shape": variable_source_dict.shape,
    "raw_long_shape": raw_long.shape,
}

{'df_final_shape': (16222, 3124),
 'variable_source_dict_shape': (3107, 8),
 'raw_long_shape': (3537664, 25)}